# 10 — LoRA on the author's SFT + GRPO CSC checkpoint

Requires the complete author baseline and train/validation candidate artifacts from **09**. This notebook does **not** train Qwen from scratch and does not repeat RL. Use a separate **training** CUDA session to avoid conflicts between vLLM and the training dependency stack.

Restore the entire experiment folder/config at the same paths. Set `CSC_CONFIG` to the saved JSON, and `RUN_CSC_GPU=1` to train. Keep train/validation databases attached for provenance checks. Do not substitute BIRD dev to obtain more pairs.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

configured = os.environ.get('SLM_PRJ_PATH')
candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/SLM-PRJ')]
REPO_DIR = next((p.resolve() for p in candidates if (p / 'research/slm_sql').is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path('/kaggle/working/SLM-PRJ')
    subprocess.run(['git', 'clone', os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git'), str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
if os.environ.get('INSTALL_CSC_DEPS', '0') == '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(REPO_DIR / 'requirements-csc-training.txt')], check=True)


In [ ]:
from scripts.csc_notebook import configure, run
CONFIG, cfg = configure(REPO_DIR)
SEED = int(os.environ.get('CSC_SEED', '42'))
RUN_GPU = os.environ.get('RUN_CSC_GPU', '0') == '1'
print({'config': str(CONFIG), 'experiment': cfg['experiment_dir'], 'seed': SEED, 'run_gpu': RUN_GPU})
run(REPO_DIR, CONFIG, 'plan', '--seed', str(SEED))


## Build verified JOIN / aggregation disagreement examples — CPU only

Use two distinct executable result groups. Keep examples only when one group matches train gold and candidate JOIN/aggregation structures differ. Train also includes already-correct leaders; swapping A/B reduces positional bias. Targets contain the verified draft SQL, with completion-only loss and no invented chain-of-thought labels.

No suitable examples is an error, not a reason to use evaluation labels. Increase limits/sample count in a separate experiment if the pilot is too small.


In [ ]:
run(REPO_DIR, CONFIG, 'pairs', '--seed', str(SEED))


## Train / resume / reuse the CSC adapter

Base weights are the exact merger commit recorded in the author baseline. Low-rank adapters only are trained; best checkpoint is chosen by held-out validation loss, not final dev EX. Matching completed training is reused. Interrupted training resumes from the last Trainer checkpoint. Overlength examples are recorded and skipped without truncating SQL targets.


In [ ]:
if RUN_GPU:
    run(REPO_DIR, CONFIG, 'train', '--seed', str(SEED))
else:
    print('Training not started. Set RUN_CSC_GPU=1 in the CUDA training session.')


In [ ]:
training_manifest = Path(cfg['experiment_dir']) / 'adapter' / 'training_manifest.json'
if training_manifest.exists():
    training = json.loads(training_manifest.read_text())
    print({key: training.get(key) for key in ('status', 'used_examples', 'skipped_context_ids', 'best_validation_checkpoint', 'metrics')})
else:
    print('No training artifact exists yet.')


## Optional: publish the verified adapter to a private Hugging Face model repo

Set Kaggle Secrets `HF_TOKEN` (write scope) and `HF_REPO_ID` (`username/repo`), then set `PUBLISH_CSC_ADAPTER=1`. This uploads only `adapter/final` and `.csc/training_manifest.json`; it does **not** upload full experiment artifacts, BIRD data or base model weights. Preserve the complete experiment folder and JSON config separately for notebook 11. Do not publish before checking the adapter manifest says `complete`.


In [ ]:
if os.environ.get('PUBLISH_CSC_ADAPTER', '0') == '1':
    from scripts.csc_notebook import publish_csc_adapter
    print(publish_csc_adapter(cfg))
else:
    print('Hub upload is off. Set PUBLISH_CSC_ADAPTER=1 only after training is complete.')
